In [ ]:
#Q2.3
import numpy as np

#Bellman equations 12x12 transition matrix. State 0 is terminal, 1-4 stochastic, 5-11 deterministic.

def compute_values(epsilon):
    """
    Computes v* and optimal policy for noise level epsilon.
    """

    T_L = np.array([
    [1,0,0,0,0,0,0,0,0,0,0,0],
    [1-epsilon,0,epsilon,0,0,0,0,0,0,0,0,0],
    [0,1-epsilon,0,epsilon,0,0,0,0,0,0,0,0],
    [0,0,1-epsilon,0,epsilon,0,0,0,0,0,0,0],
    [0,0,0,1-epsilon,0,epsilon,0,0,0,0,0,0],
    [0,0,0,0,1,0,0,0,0,0,0,0],
    [0,0,0,0,0,1,0,0,0,0,0,0],
    [0,0,0,0,0,0,1,0,0,0,0,0],
    [0,0,0,0,0,0,0,1,0,0,0,0],
    [0,0,0,0,0,0,0,0,1,0,0,0],
    [0,0,0,0,0,0,0,0,0,1,0,0],
    [0,0,0,0,0,0,0,0,0,0,1,0]
    ])

    T_R = np.array([
        [1,0,0,0,0,0,0,0,0,0,0,0],
        [epsilon,0,1-epsilon,0,0,0,0,0,0,0,0,0],
        [0,epsilon,0,1-epsilon,0,0,0,0,0,0,0,0],
        [0,0,epsilon,0,1-epsilon,0,0,0,0,0,0,0],
        [0,0,0,epsilon,0,1-epsilon,0,0,0,0,0,0],
        [0,0,0,0,0,0,1,0,0,0,0,0],
        [0,0,0,0,0,0,0,1,0,0,0,0],
        [0,0,0,0,0,0,0,0,1,0,0,0],
        [0,0,0,0,0,0,0,0,0,1,0,0],
        [0,0,0,0,0,0,0,0,0,0,1,0],
        [0,0,0,0,0,0,0,0,0,0,0,1],
        [1,0,0,0,0,0,0,0,0,0,0,0]
    ])

    R_L = np.array([0, 10-11*epsilon, -1, -1,-1, -1, -1, -1, -1, -1, -1, -1])
    R_R = np.array([0, 11*epsilon-1, -1, -1,-1, -1, -1, -1, -1, -1, -1, 10])

    v = np.zeros(12)

    for i in range(10000):
        v_new = np.zeros(12)
        for s in range(12):
            # Q-values for both actions at state s
            Q_L = R_L[s] + T_L[s] @ v
            Q_R = R_R[s] + T_R[s] @ v

            v_new[s] = max(Q_L, Q_R)

        if np.max(np.abs(v - v_new)) < 1e-12:
            print(f"Converged in {i} iterations.")
            break

        v = v_new

    # extract policy
    policy = []
    for s in range(12):
        Q_L = R_L[s] + T_L[s] @ v
        Q_R = R_R[s] + T_R[s] @ v
        print(f"State {s}: Q_L = {Q_L}, Q_R = {Q_R}")
        # if Q_L == Q_R:
            # print(f"State {s} has equal Q-values for both actions.")
        policy.append("L" if Q_L >= Q_R else "R") #ties broken aribitrarily


    return v, policy

v_02, pi_02 = compute_values(0.2)
print("Value function ε=0.2:", v_02)
print("Policy ε=0.2:", pi_02)
v_045, pi_045 = compute_values(0.45)
print("Value function ε=0.45:", v_045)
print("Policy ε=0.45:", pi_045)

Converged in 71 iterations.
State 0: Q_L = 0.0, Q_R = 0.0
State 1: Q_L = 9.337243401759414, Q_R = 7.348973607037658
State 2: Q_L = 7.68621700879744, Q_R = 5.733137829911475
State 3: Q_L = 6.082111436949394, Q_R = 4.269794721406359
State 4: Q_L = 4.665689149559589, Q_R = 3.4164222873898975
State 5: Q_L = 3.6656891495586805, Q_R = 4.0
State 6: Q_L = 3.0, Q_R = 5.0
State 7: Q_L = 4.0, Q_R = 6.0
State 8: Q_L = 5.0, Q_R = 7.0
State 9: Q_L = 6.0, Q_R = 8.0
State 10: Q_L = 7.0, Q_R = 9.0
State 11: Q_L = 8.0, Q_R = 10.0
Value function ε=0.2: [ 0.          9.3372434   7.68621701  6.08211144  4.66568915  4.
  5.          6.          7.          8.          9.         10.        ]
Policy ε=0.2: ['L', 'L', 'L', 'L', 'L', 'R', 'R', 'R', 'R', 'R', 'R', 'R']
Converged in 125 iterations.
State 0: Q_L = 0.0, Q_R = 0.0
State 1: Q_L = 6.594870522341013, Q_R = 5.838175082861239
State 2: Q_L = 3.4330456052030502, Q_R = 2.952640068061255
State 3: Q_L = 1.7908151509229668, Q_R = 1.6480972721942369
State 4: Q

In [ ]:
#Q2.4
import numpy as np
import random

#seed randomness for reproducibility
random.seed(42)
np.random.seed(42)

def sigmoid(x):
  return 1/(1+np.exp(-x))

def run_MC(num_iterations, theta, epsilon, alpha):

  for _ in range(num_iterations):
    s_0 = random.randint(0,11)
    s = s_0
    grad_J = 0

    while s != 0:
      p_R = sigmoid(s - theta)
      move = np.random.binomial(size=1, n=1, p= p_R)[0]
      if move == 1:
        grad_log_pi_wrt_theta = p_R - 1
      else:
        grad_log_pi_wrt_theta = p_R

      if s in [1,2,3,4]:
        execute = np.random.binomial(size=1, n=1, p= (1-epsilon))[0]
        if (move == 1 and execute == 1) or (move == 0 and execute == 0):
          s = (s+1) % 12
        else:
          assert (move == 1 and execute == 0) or (move == 0 and execute == 1), "INVALID MOVE"
          s = (s-1) % 12
        if s == 0:
          grad_J += 10*grad_log_pi_wrt_theta
        else:
          grad_J += -1*grad_log_pi_wrt_theta
      else:
        if move == 1:
          s = (s+1) % 12
        else:
          s = (s-1) % 12

        if s == 0:
          grad_J += 10*grad_log_pi_wrt_theta
        else:
          grad_J += -1*grad_log_pi_wrt_theta

    theta += alpha*grad_J

  return theta


thetas = []
for i in range(100000):
  theta = run_MC(1000, 0, 0.45, 0.1)
  thetas.append(theta)
print(f"theta optimal = {np.mean(thetas)}")
# theta = run_MC(1000, 0, 0.45, 0.1)
# print(f"theta optimal = {theta}")

theta optimal = 2.3185782814890756


In [ ]:
def compute_vtheta(theta, epsilon, tol=1e-12, max_iter=10000):

    # build transitions for this epsilon
    T_L = np.array([
        [1,0,0,0,0,0,0,0,0,0,0,0],
        [1-epsilon,0,epsilon,0,0,0,0,0,0,0,0,0],
        [0,1-epsilon,0,epsilon,0,0,0,0,0,0,0,0],
        [0,0,1-epsilon,0,epsilon,0,0,0,0,0,0,0],
        [0,0,0,1-epsilon,0,epsilon,0,0,0,0,0,0],
        [0,0,0,0,1,0,0,0,0,0,0,0],
        [0,0,0,0,0,1,0,0,0,0,0,0],
        [0,0,0,0,0,0,1,0,0,0,0,0],
        [0,0,0,0,0,0,0,1,0,0,0,0],
        [0,0,0,0,0,0,0,0,1,0,0,0],
        [0,0,0,0,0,0,0,0,0,1,0,0],
        [0,0,0,0,0,0,0,0,0,0,1,0]
    ])

    T_R = np.array([
        [1,0,0,0,0,0,0,0,0,0,0,0],
        [epsilon,0,1-epsilon,0,0,0,0,0,0,0,0,0],
        [0,epsilon,0,1-epsilon,0,0,0,0,0,0,0,0],
        [0,0,epsilon,0,1-epsilon,0,0,0,0,0,0,0],
        [0,0,0,epsilon,0,1-epsilon,0,0,0,0,0,0],
        [0,0,0,0,0,0,1,0,0,0,0,0],
        [0,0,0,0,0,0,0,1,0,0,0,0],
        [0,0,0,0,0,0,0,0,1,0,0,0],
        [0,0,0,0,0,0,0,0,0,1,0,0],
        [0,0,0,0,0,0,0,0,0,0,1,0],
        [0,0,0,0,0,0,0,0,0,0,0,1],
        [1,0,0,0,0,0,0,0,0,0,0,0]
    ])

    R_L = np.array([0, 10-11*epsilon, -1, -1,-1, -1, -1, -1, -1, -1, -1, -1])
    R_R = np.array([0, 11*epsilon-1, -1, -1,-1, -1, -1, -1, -1, -1, -1, 10])

    v = np.zeros(12)

    for it in range(max_iter):
        v_new = np.zeros(12)
        for s in range(12):

            pR = sigmoid(s - theta)
            pL = 1 - pR

            Q_L = R_L[s] + T_L[s] @ v
            Q_R = R_R[s] + T_R[s] @ v

            # Bellman update under fixed stochastic policy
            v_new[s] = pL * Q_L + pR * Q_R

        # convergence check
        if np.max(np.abs(v_new - v)) < tol:
            print(f"Converged in {it} iterations")
            break

        v = v_new.copy()

    return v

epsilon = 0.45

vtheta1 = compute_vtheta(2.3, epsilon)
vtheta2 = compute_vtheta(3.5, epsilon)
print(vtheta1)
print(vtheta2)
print(sum(vtheta1))
print(sum(vtheta2))

Converged in 134 iterations
Converged in 145 iterations
[0.         5.96539623 2.4728418  0.93010159 1.42709445 3.72396357
 4.92444639 5.97367289 6.99053751 7.99664367 8.99887802 9.99969841]
[0.         6.19539587 2.68548503 0.83511662 0.94204819 2.99605436
 4.67749525 5.89760132 6.96464271 7.98760547 8.99587285 9.99889216]
59.42440508437168
58.176209825554444
